# Human vs Machine Translation Detection Codebase

This Jupyter Notebook is designed to facilitate the detection and analysis of differences between human and machine-generated translations. It leverages the OpenWHO dataset from Hugging Face to extract human translations for high-resource languages (Chinese, French, English and Spanish), processes and cleans the data, and combines it with machine-generated translations from external text files.

The notebook consists of the following key components:

1. **Data Loading and Preparation**: Load subsets of the OpenWHO dataset, clean the data, and generate a CSV file containing human translations alongside source texts and language labels.

2. **Integration of Machine Translations**: Read machine-generated translations from text files and append them as new columns to the existing CSV, creating a consolidated dataset for comparative analysis.

3. **Quality Analysis**: The resulting dataset enables quality evaluation and detection tasks, such as distinguishing between human and machine translations using various metrics or models.

Ensure you have set up your Hugging Face token (`HF_TOKEN`) as an environment variable before running the code. The notebook assumes access to the required datasets and text files in specified directories.

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

login(token=os.getenv("HF_TOKEN")) # Make sure to set your Hugging Face token in the environment

OUTPUT_DIR = "openwho_csvs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ROWS_PER_LANGUAGE = 200

# OpenWHO configs
# Uncomment English and comment out Spanish for generating Spanish Evaluation set
HIGH_RESOURCE = [
    ("sent__zho", "Chinese"),
    ("sent__fra", "French"),
    # ("sent__eng", "English"),
    ("sent__spa", "Spanish"),
]



In [8]:

def load_first_n_rows(config_name: str, language_name: str, n: int = 200) -> pd.DataFrame:
    print(f"Loading {config_name} ({language_name})...")

    ds = load_dataset("raphaelmerx/openwho", config_name)["train"]

    if len(ds) < n:
        raise ValueError(f"{config_name} has only {len(ds)} rows, need {n}")

    cols = set(ds.column_names)

    if {"src", "trg"}.issubset(cols):
        src_col, trg_col = "src", "trg"

    elif {"source", "target"}.issubset(cols):
        src_col, trg_col = "source", "target"

    elif {"src_text", "tgt_text"}.issubset(cols):
        src_col, trg_col = "src_text", "tgt_text"

    elif "translation" in cols:
        sample = ds[0]["translation"]
        keys = list(sample.keys())
        if len(keys) < 2:
            raise ValueError(f"Unsupported translation dict format for {config_name}: {sample}")

        src_key, trg_key = keys[0], keys[1]

        english_text = [row["translation"][src_key] for row in ds.select(range(n))]
        target_text = [row["translation"][trg_key] for row in ds.select(range(n))]

        df = pd.DataFrame({
            "human translation": english_text,
            "source text": target_text,
            "source language": [language_name] * n,
        })
        return clean_df(df, n, config_name)

    else:
        raise ValueError(f"Unknown column format for {config_name}: {ds.column_names}")

    ds_small = ds.select(range(n))

    df = pd.DataFrame({
        "human translation": ds_small[src_col],
        "source text": ds_small[trg_col],
        "source language": [language_name] * n,
    })

    return clean_df(df, n, config_name)


def clean_df(df: pd.DataFrame, n: int, config_name: str) -> pd.DataFrame:
    df = df.dropna(subset=["human translation", "source text"]).copy()

    df["human translation"] = df["human translation"].astype(str).str.strip()
    df["source text"] = df["source text"].astype(str).str.strip()

    df = df[
        (df["human translation"] != "") &
        (df["source text"] != "")
    ].reset_index(drop=True)

    if len(df) < n:
        raise ValueError(
            f"{config_name} has only {len(df)} valid rows after cleaning"
        )

    return df.iloc[:n].copy()


def build_csv(group_configs, output_path):
    parts = []

    for config_name, language_name in group_configs:
        df_lang = load_first_n_rows(config_name, language_name, ROWS_PER_LANGUAGE)
        parts.append(df_lang)
        print(f"Added {len(df_lang)} rows for {language_name}")

    combined = pd.concat(parts, ignore_index=True)

    combined.to_csv(output_path, index=False, encoding="utf-8")

    print(f"\nSaved {len(combined)} rows to {output_path}")
    print("\nCounts:")
    print(combined["source language"].value_counts())

    return combined




In [ ]:
# Run
output_file = os.path.join(OUTPUT_DIR, "high_resource_openwho_600.csv")
df = build_csv(HIGH_RESOURCE, output_file)

print("\nDone.")

# Combining Translated Text for Quality Analysis

The code below is used to combine translated text from multiple sources (human translations and machine-generated translations) into a single CSV file. This consolidated dataset will be used for quality analysis to compare and evaluate the differences between human and machine translations.

In [ ]:
import os
import pandas as pd


def extract_column_name(filename):
    """
    Extract column name from filename.
    Example: openwho_test_gemini.txt -> gemini
    """
    base = os.path.basename(filename)
    name = os.path.splitext(base)[0]  
    return name.split("_")[-1]       


def read_txt_file(filepath):
    """
    Read txt file and return cleaned lines.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]
    
    # Remove empty lines if any
    lines = [line for line in lines if line]
    return lines


def add_txt_columns_to_csv(csv_path, txt_folder, output_path):
    # Load CSV
    df = pd.read_csv(csv_path)
    num_rows = len(df)

    print(f"CSV loaded with {num_rows} rows")

    # Process each txt file
    for file in os.listdir(txt_folder):
        if file.endswith(".txt"):
            file_path = os.path.join(txt_folder, file)

            column_name = extract_column_name(file)
            lines = read_txt_file(file_path)

            print(f"Processing {file} → column '{column_name}'")

            # Validate length
            if len(lines) != num_rows:
                raise ValueError(
                    f"Row mismatch in {file}: "
                    f"{len(lines)} lines vs {num_rows} CSV rows"
                )

            # Add column
            df[column_name] = lines

    # Save new CSV
    df.to_csv(output_path, index=False)
    print(f"\nSaved updated CSV to: {output_path}")

In [ ]:
csv_file = "" # Set the path to generated CSV file above 
txt_folder = " " # Set the path to the folder containing the txt files with machine translation outputs
output_file = " output.csv" # Set the path for the output CSV file with all translations

add_txt_columns_to_csv(csv_file, txt_folder, output_file)